### GRASS setup

In [ ]:
import os, sys, shutil, subprocess, re, glob 
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make GRASS Python packages importable.
sys.path.append(
    subprocess.check_output(["grass85", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj

In [ ]:
GISDBASE = "/home/pregmi3/Desktop/helene_nov_2025/data_now_17_2025/Notebooks"

In [ ]:
#project_path = "DEM_generation"

In [ ]:
session = gj.init(GISDBASE, location="DEM_generation", mapset="for_codem")

In [ ]:
#gs.run_command("g.mapset", flags="c", mapset="registration", overwrite=True)

In [ ]:
!g.mapset -p

### Creating profiles for multitemporal DEMs 

#### Generate registration points

In [ ]:
!g.region raster="dtm_2017" res=1

In [ ]:
map = gj.InteractiveMap()
map.add_raster("Ortho_aerial_2024_rgb@DTM_DSM")
#map.add_raster("lidar_2024_dsm_relief_filled@DTM_DSM")
map.add_vector("new_profile")
map.add_vector("more_stable_pts")
#map.add_vector("dsm_overlap_poly")
map.show()

#### generate before profile

In [ ]:
#!g.list type=vector

In [ ]:
line_vec = "vertical_line_long"   # <-- change this

rasters = [
    "dtm_2017",
    "dtm_2020_filled",
    "dtm_2024_filled",
    "cap_dtm_2024_filled",
]

# Export ALL features as WKT and parse ALL vertices (fixes the "always same line" issue)
wkt_txt = gs.read_command("v.out.ascii", input=line_vec, format="wkt").strip()
pairs = re.findall(r"(-?\d+(?:\.\d+)?)\s+(-?\d+(?:\.\d+)?)", wkt_txt)

if len(pairs) < 2:
    raise RuntimeError(f"Could not extract coordinates from {line_vec}")

# Remove consecutive duplicate vertices (optional but helps)
coords_list = []
prev = None
for x, y in pairs:
    if prev != (x, y):
        coords_list.append(f"{x},{y}")
        prev = (x, y)

coords = ",".join(coords_list)

print("Vertices:", len(coords_list))
print("First:", coords_list[0], " Last:", coords_list[-1])

out_dir = Path(f"./multitemporal_{line_vec}")
out_dir.mkdir(parents=True, exist_ok=True)

for r in rasters:
    out_csv = out_dir / f"profile_{r}.csv"
    gs.run_command(
        "r.profile",
        input=r,
        coordinates=coords,
        resolution=1,
        output=str(out_csv),
        flags="g",
        overwrite=True,
    )
    print("Wrote:", out_csv)

print("Done ->", out_dir)

In [ ]:
#line_vec = "profile_test5"  # must match above
files = sorted(glob.glob(f"./multitemporal_{line_vec}/profile_*.csv"))
if not files:
    raise FileNotFoundError("No profile CSVs found for this line_vec.")

def load_profile(f):
    df = pd.read_csv(f, header=None, comment="#", sep=None, engine="python")
    if df.shape[1] == 1:
        df = df[0].astype(str).str.strip()
        df = df[(df != "") & (~df.str.startswith("#"))].str.split(r"[,\s]+", expand=True)
    df = df.apply(pd.to_numeric, errors="coerce").dropna(how="all")

    if df.shape[1] >= 3:
        x, y, v = df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy(), df.iloc[:,-1].to_numpy()
        d = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
        return d, v
    return df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy()

plt.figure(figsize=(18, 6))
for f in files:
    d, v = load_profile(f)
    label = re.sub(r"^profile_|\.csv$", "", os.path.basename(f))
    plt.plot(d, v, label=label)

plt.xlabel("Distance (m)")
plt.ylabel("Elevation")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)

plt.savefig(f"vertical_profile_{line_vec}.png", dpi=300, bbox_inches="tight")
plt.show()

#### Coregistration

In [ ]:
from pandas import json_normalize
PTS   = "more_stable_pts"
R_REF = "dtm_2017"

TARGETS = [
    "dtm_2020_filled",
    "dtm_2024_filled",
    "cap_dtm_2024_filled",
   ]

SUFFIX = "_vref2017_rmdse"

def signed_rmdse_bias(dem1: str, dem2: str, control_points: str):
    data = gs.parse_command(
        "r.what",
        flags="n",
        map=f"{dem2},{dem1}",
        points=control_points,
        format="json",
    )
    df = json_normalize(data)

    dem1_elev = f"{dem1}.value"
    dem2_elev = f"{dem2}.value"

    # make numeric + drop missing
    df[dem1_elev] = np.array(df[dem1_elev], dtype=float)
    df[dem2_elev] = np.array(df[dem2_elev], dtype=float)
    df = df[np.isfinite(df[dem1_elev]) & np.isfinite(df[dem2_elev])].copy()

    diff = df[dem2_elev] - df[dem1_elev]            # target - reference
    med  = float(np.median(diff))                   # SIGN comes from this
    mag  = float(np.sqrt(np.median(diff**2)))       # RMdSE magnitude (always +)

    sgn = 1.0 if med >= 0 else -1.0                 # if target above -> +, below -> -
    signed_bias = sgn * mag                         # signed RMdSE shift
    return signed_bias, mag, med

def outname(raster, suffix):
    return f"{raster.split('@')[0]}{suffix}"

gs.run_command("g.region", raster=R_REF)

results = []
for tgt in TARGETS:
    out = outname(tgt, SUFFIX)

    signed_bias, mag, med = signed_rmdse_bias(R_REF, tgt, PTS)
    # apply: corrected = target - signed_bias
    #  - if target above (med>0): subtract +mag
    #  - if target below (med<0): subtract -mag  => add mag
    gs.mapcalc(f"{out} = {tgt} - ({signed_bias})", overwrite=True)

    new_signed_bias, new_mag, new_med = signed_rmdse_bias(R_REF, out, PTS)

    print(f"\n--- {tgt} -> {out} ---")
    print(f"median(diff)={med:.4f}  RMdSE_mag={mag:.4f}  signed_bias={signed_bias:.4f}")
    print(f"after: median(diff)={new_med:.4f}  RMdSE_mag={new_mag:.4f}  signed_bias={new_signed_bias:.4f}")

    results.append((tgt, out, signed_bias, mag, med, new_mag, new_med))

print("\nDONE.")

In [ ]:
rename_pairs = {
    "dtm_2020_filled_vref2017_rmdse":                         "lidar_2020_regis",
    "dtm_2024_filled_vref2017_rmdse":   "lidar_2024_regis",
    "cap_dtm_2024_filled_vref2017_rmdse":                "cap_sfm_regis"}

for old, new in rename_pairs.items():
    print(f"Renaming {old} -> {new}")
    gs.run_command("g.rename", raster=f"{old},{new}", overwrite=True)

# now your plotting list can be:
rasters = [
    "dtm_2017",
    
    "lidar_2020_regis",
    
    "lidar_2024_regis",
   
    "cap_sfm_regis",
    
]
print(rasters)

In [ ]:
#line_vec = "profile_test5"

rasters = [
    "dtm_2017",
    
    "lidar_2020_regis",
    
    "lidar_2024_regis",
   
    "cap_sfm_regis",
    
]

# get line vertices as WKT, extract x y pairs
wkt_txt = gs.read_command("v.out.ascii", input=line_vec, format="wkt").strip()
wkt_line = wkt_txt.splitlines()[0]
pairs = re.findall(r"(-?\d+(?:\.\d+)?)\s+(-?\d+(?:\.\d+)?)", wkt_line)
coords = ",".join([f"{x},{y}" for x, y in pairs])

out_dir = Path("./multitemporal_registered")
out_dir.mkdir(parents=True, exist_ok=True)

for r in rasters:
    out_csv = out_dir / f"profile_{r}.csv"
    gs.run_command(
        "r.profile",
        input=r,
        coordinates=coords,
        resolution=1,
        output=str(out_csv),
        flags="g",
        overwrite=True,
    )
    print("Wrote:", out_csv)

In [ ]:
files = sorted(glob.glob("./multitemporal_registered/profile_*.csv"))
if not files: raise FileNotFoundError("No profile_*.csv found")

def load_profile(f):
    df = pd.read_csv(f, header=None, comment="#", sep=None, engine="python")
    if df.shape[1] == 1:
        df = df[0].astype(str).str.strip()
        df = df[(df != "") & (~df.str.startswith("#"))].str.split(r"[,\s]+", expand=True)
    df = df.apply(pd.to_numeric, errors="coerce").dropna(how="all")

    if df.shape[1] >= 3:  # x,y,value -> distance
        x, y, v = df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy(), df.iloc[:,-1].to_numpy()
        d = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
        return d, v
    return df.iloc[:,0].to_numpy(), df.iloc[:,1].to_numpy()

plt.figure(figsize=(18, 6))
for f in files:
    d, v = load_profile(f)
    label = re.sub(r"^profile_|\.csv$", "", os.path.basename(f))
    plt.plot(d, v, label=label)

plt.xlabel("Distance (m)")
plt.ylabel("Elevation")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)
plt.savefig("Multitemporal_registered_vertical.png", dpi=300, bbox_inches="tight")
plt.show()